# Clean Raw Data

**Input:** `data_csv/zillow_full_raw.csv`

**Output:** `data_csv/zillow_clean.csv`

Mục tiêu của notebook này là chuẩn hóa dữ liệu raw/enriched từ Zillow, chọn các cột cần thiết, đổi tên cột về format ngắn gọn và lưu ra file trung gian `zillow_clean.csv`.

In [1]:

import sys
assert sys.version_info >= (3, 5)


import warnings
warnings.filterwarnings('ignore')

import sklearn
assert sklearn.__version__ >= "0.20"

# import thư viện
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)
import seaborn as sns

%matplotlib inline

## 2. Chọn cột và chuẩn hóa tên biến

In [2]:
df = pd.read_csv(('../data_csv/zillow_full_raw.csv'),sep=',',on_bad_lines='skip',  engine="python")
df.head()


,zpid,palsId,rawHomeStatusCd,marketingStatusSimplifiedCd,imgSrc,hasImage,detailUrl,statusType,statusText,price,...,nearest_city_san_francisco.1,nearest_city_san_jose.1,nearest_city_nan.1,nearest_airport_LAX.1,nearest_airport_OAK.1,nearest_airport_SAN.1,nearest_airport_SFO.1,nearest_airport_SJC.1,nearest_airport_SMF.1,nearest_airport_nan.1
0,2.073568e+09,17327010_26844339,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/baa91158903...,True,/homedetails/8721-W-Sunset-Plaza-Ter-West-Holl...,FOR_SALE,House for sale,"$4,980,000",...,False,False,False,True,False,False,False,False,False,False
1,1.987515e+07,31493001_226002630,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/60873898547...,True,/homedetails/23649-Aetna-St-Woodland-Hills-CA-...,FOR_SALE,House for sale,"$1,215,000",...,False,False,False,True,False,False,False,False,False,False
2,2.053261e+07,17327010_26846711,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/8d2b3e54fc2...,True,/homedetails/2350-Benedict-Canyon-Dr-Beverly-H...,FOR_SALE,House for sale,"$2,629,000",...,False,False,False,True,False,False,False,False,False,False
3,NaN,NaN,NaN,NaN,https://photos.zillowstatic.com/fp/07b915b7231...,True,/b/645-w-9th-st-los-angeles-ca-CkBBCq/,FOR_SALE,For Rent,"From $389,000",...,False,False,False,True,False,False,False,False,False,False
4,2.093480e+07,17327006_26843895,ForSale,For Sale by Agent,https://photos.zillowstatic.com/fp/f95fee73996...,True,/homedetails/1161-W-67th-St-Los-Angeles-CA-900...,FOR_SALE,Multi-family home for sale,"$400,000",...,False,False,False,True,False,False,False,False,False,False


In [3]:
# Số lượng data
len(df)

5757

In [4]:
nan_value = float("NaN")
df.replace("0", nan_value, inplace=True)
# Thống kê  tỷ lệ phần trăm giá trị null trong các cột
(df.isnull().sum() / df.isnull().count()).sort_values(ascending=False)

plid                                    1.000000
buildingName                            1.000000
listPriceIncludesRequiredMonthlyFees    0.999826
style                                   0.999826
canSaveBuilding                         0.999826
                                          ...   
nri_expected_annual_loss_score          0.000000
nri_overall_risk_rating                 0.000000
nri_overall_risk_score                  0.000000
has3DModel                              0.000000
nearest_airport_nan.1                   0.000000
Length: 227, dtype: float64

In [5]:
#Đổi tất cả giá trị -1 thành NaN
df.replace(-1, np.nan, inplace=True)

In [6]:
# Drop cột null > 50%
thresh = len(df) * 0.5
df.dropna(thresh = thresh, axis = 1, inplace = True)

In [7]:
#Xóa các cột có giá trị giống hệt nhau
df = df.loc[:, ~df.T.duplicated()]

In [8]:
#Xem phần trăm dữ liệu bị missing còn lại
(df.isnull().sum() / df.isnull().count()).sort_values(ascending=False)

zestimate_to_price_ratio      0.330554
hdpData.homeInfo.zestimate    0.326038
lot_to_living_ratio           0.291124
brokerName                    0.229807
flexFieldText                 0.216780
                                ...   
county_fips                   0.000000
tract_fips                    0.000000
tract_geoid                   0.000000
tract_arealand_km2            0.000000
osm_lon_grid                  0.000000
Length: 118, dtype: float64

In [9]:
df.columns


Index(['zpid', 'palsId', 'rawHomeStatusCd', 'marketingStatusSimplifiedCd',
       'imgSrc', 'hasImage', 'detailUrl', 'statusType', 'statusText', 'price',
       ...
       'nearest_city_san_diego', 'nearest_city_san_francisco',
       'nearest_city_san_jose', 'nearest_airport_LAX', 'nearest_airport_OAK',
       'nearest_airport_SFO', 'nearest_airport_SJC', 'nearest_airport_SMF',
       'osm_lat_grid', 'osm_lon_grid'],
      dtype='object', length=118)

In [10]:
len(df.columns)

118

In [11]:
df["target_price"] = pd.to_numeric(
    df["hdpData.homeInfo.price"],
    errors="coerce"
)

In [12]:

# Chuẩn hóa living area về sqft

df["living_area_sqft"] = pd.to_numeric(
    df["area"],
    errors="coerce"
)

#  Chuẩn hóa lot về sqft

lot_value = pd.to_numeric(
    df["hdpData.homeInfo.lotAreaValue"],
    errors="coerce"
)

lot_unit = df["hdpData.homeInfo.lotAreaUnit"].astype(str).str.lower()

df["lot_area_sqft"] = np.where(
    lot_unit.str.contains("acre", na=False),
    lot_value * 43560,
    lot_value
)

In [13]:
df["lot_to_living_ratio"] = (
    df["lot_area_sqft"] / df["living_area_sqft"].replace(0, np.nan)
)

df["beds_per_bath"] = (
    pd.to_numeric(df["beds"], errors="coerce")
    / pd.to_numeric(df["baths"], errors="coerce").replace(0, np.nan)
)

In [14]:
# Chọn cột bằng tên gốc trong file
keep_cols = [
    # Target
    "target_price",

    # Basic house features
    "beds",
    "baths",
    "area",
    "lot_area_sqft",
    "lot_to_living_ratio",
    "beds_per_bath",

    # Home type one-hot
    "home_type_CONDO",
    "home_type_LOT",
    "home_type_MANUFACTURED",
    "home_type_MULTI_FAMILY",
    "home_type_SINGLE_FAMILY",
    "home_type_TOWNHOUSE",

    # Location coordinates
    "latLong.latitude",
    "latLong.longitude",

    # Socioeconomic features
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "population",
    "bachelor_or_higher_rate",
    "median_gross_rent",
    "owner_occupied_rate",
    "population_density_per_km2",

    # Risk features
    "nri_overall_risk_score",
    "nri_expected_annual_loss_score",
    "nri_social_vulnerability_score",
    "nri_resilience_score",
    "wildfire_risk_score",
    "earthquake_risk_score",
    "heatwave_risk_score",

    # Distance features
    "distance_to_nearest_major_ca_city_km",
    "distance_to_nearest_major_ca_airport_km",
    "distance_to_ca_coast_approx_km",
]

# 2. Chỉ giữ cột nào thật sự tồn tại trong df
keep_cols = [col for col in keep_cols if col in df.columns]

# 3. Tạo df_model trước
df_model = df[keep_cols].copy()

# 4. Rename
rename_map = {
    "target_price": "price",

    "beds": "bed",
    "baths": "bath",
    "area": "living",
    "lot_area_sqft": "lot_sqft",
    "lot_to_living_ratio": "lot_living",
    "beds_per_bath": "bed_bath",

    "home_type_CONDO": "type_condo",
    "home_type_LOT": "type_lot",
    "home_type_MANUFACTURED": "type_manufactured",
    "home_type_MULTI_FAMILY": "type_multi",
    "home_type_SINGLE_FAMILY": "type_single",
    "home_type_TOWNHOUSE": "type_townhouse",

    "latLong.latitude": "lat",
    "latLong.longitude": "long",

    "median_household_income": "income",
    "poverty_rate": "poverty",
    "unemployment_rate": "unemployment",
    "population": "pop",
    "bachelor_or_higher_rate": "bachelor",
    "median_gross_rent": "rent",
    "owner_occupied_rate": "owner_occ",
    "population_density_per_km2": "pop_density",

    "nri_overall_risk_score": "risk_overall",
    "nri_expected_annual_loss_score": "risk_loss",
    "nri_social_vulnerability_score": "risk_social",
    "nri_resilience_score": "risk_resilience",
    "wildfire_risk_score": "risk_fire",
    "earthquake_risk_score": "risk_earthquake",
    "heatwave_risk_score": "risk_heat",

    "distance_to_nearest_major_ca_city_km": "dist_city",
    "distance_to_nearest_major_ca_airport_km": "dist_airport",
    "distance_to_ca_coast_approx_km": "dist_coast",
}

df_model.rename(columns=rename_map, inplace=True)

# 5. Kiểm tra kết quả
print("Shape:", df_model.shape)
print(df_model.columns.tolist())


Shape: (5757, 33)
['price', 'bed', 'bath', 'living', 'lot_sqft', 'lot_living', 'bed_bath', 'type_condo', 'type_lot', 'type_manufactured', 'type_multi', 'type_single', 'type_townhouse', 'lat', 'long', 'income', 'poverty', 'unemployment', 'pop', 'bachelor', 'rent', 'owner_occ', 'pop_density', 'risk_overall', 'risk_loss', 'risk_social', 'risk_resilience', 'risk_fire', 'risk_earthquake', 'risk_heat', 'dist_city', 'dist_airport', 'dist_coast']


In [15]:
# Lưu file trung gian cho bước 02
df_model.to_csv("../data_csv/zillow_clean.csv", index=False, encoding="utf-8-sig")

print(f"Saved zillow_clean.csv with shape: {df_model.shape}")
print(df_model.columns.tolist())

Saved zillow_clean.csv with shape: (5757, 33)
['price', 'bed', 'bath', 'living', 'lot_sqft', 'lot_living', 'bed_bath', 'type_condo', 'type_lot', 'type_manufactured', 'type_multi', 'type_single', 'type_townhouse', 'lat', 'long', 'income', 'poverty', 'unemployment', 'pop', 'bachelor', 'rent', 'owner_occ', 'pop_density', 'risk_overall', 'risk_loss', 'risk_social', 'risk_resilience', 'risk_fire', 'risk_earthquake', 'risk_heat', 'dist_city', 'dist_airport', 'dist_coast']
